# Chapter 23: Time Series Forecasting (Production Mindset)

## Learning Objectives\n- Use walk-forward validation correctly\n- Build leakage-safe lag/rolling features\n- Compare against strong naive baselines\n- Define retraining and monitoring policy

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 23: Time Series Forecasting (Production Mindset)

## Why this chapter matters
Time series work is less about model complexity and more about proper temporal validation, leakage control, and monitoring drift.

## Learning goals
1. Build lag and rolling-window features safely.
2. Use walk-forward validation.
3. Quantify forecast uncertainty.
4. Design retraining policy.

## Step 1: Problem framing
Define:
- forecast horizon (`t+1`, `t+7`, etc.)
- prediction frequency (hourly/daily/weekly)
- acceptable error metric (MAE, sMAPE, MASE)

## Step 2: Safe feature engineering
Allowed features at time `t`:
- lag values (`y_{t-1}`, `y_{t-7}`)
- rolling means computed only from past values
- calendar features (day-of-week, month, holiday)

Forbidden:
- future known target information
- global normalization using all timestamps

## Step 3: Walk-forward backtesting
1. train on `[0..k]`, validate on `[k+1..k+h]`
2. expand or slide training window
3. repeat and aggregate metrics

## Step 4: Baselines before complex models
Always benchmark against:
1. naive (`y_t = y_{t-1}`)
2. seasonal naive (`y_t = y_{t-s}`)

If advanced model cannot beat these consistently, stop and debug.

## Step 5: Production checklist
- late/missing data handling
- retrain schedule policy
- error monitoring dashboard
- concept drift alarms

## Assignment
Implement a walk-forward evaluator using your own lag-feature regressor and compare with naive baseline.



In [ ]:
from __future__ import annotations

from typing import Callable, List


def mae(y_true: List[float], y_pred: List[float]) -> float:
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)


def seasonal_naive(train: List[float], horizon: int, season: int = 7) -> List[float]:
    return [train[-season + (i % season)] for i in range(horizon)]


def walk_forward(
    series: List[float],
    min_train_size: int,
    horizon: int,
    forecaster: Callable[[List[float], int], List[float]],
) -> float:
    preds = []
    truth = []

    start = min_train_size
    while start + horizon <= len(series):
        train = series[:start]
        test = series[start : start + horizon]
        f = forecaster(train, horizon)
        preds.extend(f)
        truth.extend(test)
        start += horizon

    return mae(truth, preds)


if __name__ == "__main__":
    # Synthetic weekly seasonal data
    data = [10, 12, 13, 11, 9, 8, 7] * 8
    score = walk_forward(
        series=data,
        min_train_size=21,
        horizon=7,
        forecaster=lambda tr, h: seasonal_naive(tr, h, season=7),
    )
    print("Walk-forward MAE:", score)


## Checkpoint\n\n1. You can explain why random split is wrong for temporal data.\n2. You can compute walk-forward MAE.\n3. You can define seasonal-naive baseline use cases.

## Guided Exercise\nImplement a moving-average forecaster and compare MAE vs seasonal naive.

In [ ]:
# TODO (Student Exercise)
def moving_average_forecaster(train, horizon, window=7):
    avg = sum(train[-window:]) / window
    return [avg] * horizon

data = [10,12,13,11,9,8,7] * 8
score = walk_forward(data, 21, 7, lambda tr,h: moving_average_forecaster(tr,h,window=7))
print('MAE:', score)

## Quick Quiz\n\n**Q1.** What is leakage in time series context?\n\n**Answer:** Using information from future timestamps in training features.\n\n**Q2.** Why evaluate with walk-forward?\n\n**Answer:** It mirrors real deployment where future is unseen.\n\n**Q3.** When is seasonal naive strong?\n\n**Answer:** When series has stable seasonal pattern.\n

## Homework\nCreate a backtesting report with at least 3 folds and include error by horizon.